# Qwen3.5-9B on a FREE Kaggle T4 (llama.cpp + GGUF)
Before running, open the right sidebar -> **Session options** and set:
- **Accelerator**: `GPU T4 x2`
- **Internet**: `On` (needs a phone-verified Kaggle account)
- **Persistence**: `Files only` (keeps the model + built binary in `/kaggle/working` for the next session)

Then run the cells **top to bottom**.

What this does: downloads the real `unsloth/Qwen3.5-9B-GGUF` weights, builds `llama-server` with CUDA, starts an
OpenAI-compatible server with schema-constrained JSON output, and prints a public URL + the exact `.env` lines.
Quantized (Q4/Q5) is slightly weaker than full BF16, so evals measured here are *GGUF* numbers.


In [ ]:
import subprocess, requests
try:
    print(subprocess.check_output(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv'], text=True))
except Exception:
    raise SystemExit('No GPU: Session options -> Accelerator -> GPU T4 x2, then restart the session.')
try:
    requests.get('https://huggingface.co', timeout=10)
except Exception:
    raise SystemExit('No internet: Session options -> Internet -> On (verify your phone number in Kaggle settings first).')
!df -h /kaggle/working | tail -1
!free -g | head -2


## 1. Settings


In [ ]:
import secrets, os
QUANT = 'Q4_K_M'          # Q4_K_M (5.3 GB, fastest) | Q5_K_M (6.1 GB, better) | Q6_K (7.0 GB)
CTX = 16384               # context window; lower to 8192 if you hit out-of-memory
GPUS = '0'                # one T4 is enough for these quants; '0,1' spreads the model over both T4s
# Stable key: Add-ons -> Secrets -> add LLM_API_KEY and attach it to this notebook. Otherwise a new random key each session.
try:
    from kaggle_secrets import UserSecretsClient
    API_KEY = UserSecretsClient().get_secret('LLM_API_KEY')
    print('API key: from Kaggle secret LLM_API_KEY')
except Exception:
    API_KEY = secrets.token_urlsafe(24)   # protects your endpoint; goes into apps/ai/.env as LLM_API_KEY
    print('API key: random for this session')
BASE = '/kaggle/working/qwen'   # survives restarts when Persistence = Files only
os.makedirs(BASE, exist_ok=True)
print('QUANT', QUANT, '| CTX', CTX, '| GPUS', GPUS, '| cache dir', BASE)


## 2. Download the real model (skipped if already cached)


In [ ]:
!pip -q install -U huggingface_hub hf_transfer
os.environ['HF_HUB_ENABLE_HF_TRANSFER'] = '1'
from huggingface_hub import hf_hub_download
import time, os
t = time.time()
MODEL_PATH = hf_hub_download('unsloth/Qwen3.5-9B-GGUF', f'Qwen3.5-9B-{QUANT}.gguf', local_dir=BASE)
print(MODEL_PATH, round(os.path.getsize(MODEL_PATH) / 1e9, 2), 'GB in', round(time.time() - t), 's')


## 3. Build llama.cpp with CUDA (about 6-10 min the first time; cached in `/kaggle/working` afterwards)


In [ ]:
import os, shutil, subprocess
try:
    ARCH = subprocess.check_output(['nvidia-smi', '--query-gpu=compute_cap', '--format=csv,noheader'], text=True).split()[0].replace('.', '')
except Exception:
    ARCH = '75'   # T4
BIN_DIR = f'{BASE}/bin-sm{ARCH}'   # per GPU architecture, so a T4 build is never reused on a P100
BIN = f'{BIN_DIR}/llama-server'
if os.path.exists(BIN):
    print('using cached binary', BIN)
else:
    !rm -rf /tmp/llama.cpp && git clone --depth 1 https://github.com/ggml-org/llama.cpp /tmp/llama.cpp
    !cd /tmp/llama.cpp && cmake -B build -DGGML_CUDA=ON -DCMAKE_CUDA_ARCHITECTURES={ARCH} -DLLAMA_CURL=OFF -DCMAKE_BUILD_TYPE=Release -DLLAMA_BUILD_TESTS=OFF -DLLAMA_BUILD_EXAMPLES=OFF
    !cd /tmp/llama.cpp && cmake --build build --config Release -j 4 --target llama-server
    assert os.path.exists('/tmp/llama.cpp/build/bin/llama-server'), 'build failed - read the cmake output above'
    os.makedirs(BIN_DIR, exist_ok=True)
    for f in os.listdir('/tmp/llama.cpp/build/bin'):
        if f.endswith('.so'):
            shutil.copy(f'/tmp/llama.cpp/build/bin/{f}', BIN_DIR)
    shutil.copy('/tmp/llama.cpp/build/bin/llama-server', BIN)
    os.chmod(BIN, 0o755)
print('binary ready')


## 4. Start the server
`--jinja` uses the model's own chat template; thinking is switched OFF (our client also sends `enable_thinking=false`).


In [ ]:
import subprocess, time, requests
env = dict(os.environ, CUDA_VISIBLE_DEVICES=GPUS,
           LD_LIBRARY_PATH=f'{BIN_DIR}:/usr/local/cuda/lib64:/usr/local/nvidia/lib64:' + os.environ.get('LD_LIBRARY_PATH', ''))
cmd = [BIN, '-m', MODEL_PATH, '--host', '0.0.0.0', '--port', '8001', '-ngl', '99', '-c', str(CTX),
       '--api-key', API_KEY, '-a', 'Qwen/Qwen3.5-9B', '--jinja', '-np', '2', '-fa', 'on',
       '--chat-template-kwargs', '{"enable_thinking": false}']
log = open('/tmp/llama-server.log', 'w')
server = subprocess.Popen(cmd, stdout=log, stderr=subprocess.STDOUT, env=env)
for i in range(120):
    try:
        if requests.get('http://localhost:8001/health', timeout=2).status_code == 200:
            print('server is up after', i * 2, 's'); break
    except Exception:
        pass
    time.sleep(2)
else:
    print('NOT UP - last log lines:'); print(open('/tmp/llama-server.log').read()[-3000:])


## 5. Self-test: schema-constrained JSON must come back valid


In [ ]:
import json, requests
schema = {'type': 'object', 'properties': {'signals': {'type': 'array', 'items': {'type': 'object', 'properties': {'kind': {'type': 'string'}, 'quote': {'type': 'string'}}, 'required': ['kind', 'quote']}}}, 'required': ['signals']}
body = {'model': 'Qwen/Qwen3.5-9B', 'temperature': 0, 'max_tokens': 300,
        'messages': [{'role': 'system', 'content': 'Return JSON only.'}, {'role': 'user', 'content': '<message>We are also looking at Gainsight.</message> List signals.'}],
        'response_format': {'type': 'json_schema', 'json_schema': {'name': 'x', 'schema': schema, 'strict': True}},
        'chat_template_kwargs': {'enable_thinking': False}}
t = time.time()
r = requests.post('http://localhost:8001/v1/chat/completions', json=body, headers={'Authorization': f'Bearer {API_KEY}'}, timeout=120).json()
print(r['choices'][0]['message']['content'])
print('tokens:', r['usage'], '| seconds:', round(time.time() - t, 1))
json.loads(r['choices'][0]['message']['content']); print('VALID JSON')


## 6. Public URL (Cloudflare quick tunnel, no account needed)


In [ ]:
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /tmp/cloudflared && chmod +x /tmp/cloudflared
import re
tun = subprocess.Popen(['/tmp/cloudflared', 'tunnel', '--url', 'http://localhost:8001'], stdout=open('/tmp/tunnel.log', 'w'), stderr=subprocess.STDOUT)
url = None
for _ in range(40):
    time.sleep(1)
    m = re.search(r'https://[a-z0-9-]+\.trycloudflare\.com', open('/tmp/tunnel.log').read())
    if m:
        url = m.group(0); break
print()
if url is None:
    print('NO TUNNEL URL - last log lines:'); print(open('/tmp/tunnel.log').read()[-3000:])
else:
    print('=== put these 4 lines in apps/ai/.env on your laptop ===')
    print(f'LLM_BASE_URL={url}/v1')
    print(f'LLM_API_KEY={API_KEY}')
    print('LLM_MODEL=Qwen/Qwen3.5-9B')
    print('LLM_STRUCTURED_MODE=response_format')


## 7. Keep it alive
Leave this tab open and run the next cell; it just prints the server log tail every minute. A Kaggle session stops
when idle or after ~12 h, and the free GPU quota is ~30 h per week. If it stops, re-run all cells (the download and
build are cached when Persistence = Files only) and update the `.env` URL - the tunnel URL changes every session.


In [ ]:
import time
while True:
    time.sleep(60)
    print(time.strftime('%H:%M'), open('/tmp/llama-server.log').read().strip().splitlines()[-1][:160])
